## 1. Abstract screening

### 1.1. Cholangiocarcinoma

In [ ]:
# Cholangio_Ontology

import json, re, time, hashlib
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import os
from pathlib import Path

MODEL_NAME = "gpt-5-mini-2025-08-07"          # 권장: 동세대 고정 ID(또는 스냅샷명). 필요 시 "gpt-5-mini"
SEED = 12345
TEMP = 1
TOP_P = 1
JSON_MODE = {"type": "json_object"}  # JSON 강제(지원 모델)
INPUT_1M = 0.25 
OUTPUT_1M = 2.00 
INPUT_XLSX = "./ELLMA1_cholangio_250910_abstracts.xlsx"
OUT_JSONL  = Path("./results.jsonl")   # 행 단위 즉시 기록(체크포인트)
OUT_XLSX   = Path("./results.xlsx")    # 최종 요약 저장

client = OpenAI(api_key=API_KEY, timeout=60)
# =========================================
# gpt-4-0613, gpt-5-2025-08-07, gpt-5-mini-2025-08-07 비교
# INPUT_1M = 2.5, 1.25, 0.25
# OUTPUT_1M = 10.00, 10.00, 2.00
# Ontology: Definition of Key Terms and Links to Existing Terminology; Adjuvant radiotherapy refers to radiotherapy used after curative or near-curative surgery (including R1 resection) with the purpose of reducing recurrence, and radiotherapy administered in cases of unresectable disease or palliative situations (including R2 resection) is not considered adjuvant radiotherapy. Extrahepatic cholangiocarcinoma is synonymous with Klatskin’s tumor, and extrahepatic bile duct cancer. It can be described in the literature as cholangiocarcinoma, periampullary cancer, ampulla of Vater cancer, biliary tract cancer, or bile duct cancer. However, studies of pancreatic cancer, gallbladder cancer, or intrahepatic cholangiocarcinoma are excluded. 

# System message 변수 정의
system_message = """
You are a specialized assistant for experts conducting meta-analyses and developing clinical practice guidelines.
You support users in formulating key questions, conducting abstract screening, and performing data extraction during systematic reviews.
You emphasize evidence-based decision-making and strictly follow up-to-date research methodologies and guideline development processes.

You utilize frameworks such as PICO(TS) to structure research questions and apply methodologies like PRISMA and GRADE to assess the quality of evidence.
You also help optimize literature search strategies and refine inclusion and exclusion criteria based on the user's objectives.

You respond with precise terminology and accurate information. When needed, you can search for the latest research trends and suggest the most appropriate methodological approach according to the user's context and goals.
"""

# 선별 규칙 설명 프롬프트
SCREENING_PROMPT = """
Research Hypothesis and Objectives ;This file is a list of abstracts for meta-analysis. The hypothetical question of this meta-analysis is: “In extrahepatic 
cholangiocarcinoma, does application of adjuvant radiotherapy (ART) after surgical resection has benefit on patients as compared to those did not undergo ART, 
in regard of overall survival?”. 

Ontology: Definition of Key Terms and Links to Existing Terminology; Adjuvant radiotherapy refers to radiotherapy used after curative or near-curative surgery (including R1 resection) with the purpose of reducing recurrence, and radiotherapy administered in cases of unresectable disease or palliative situations (including R2 resection) is not considered adjuvant radiotherapy. Extrahepatic cholangiocarcinoma is synonymous with Klatskin’s tumor, and extrahepatic bile duct cancer. It can be described in the literature as cholangiocarcinoma, periampullary cancer, ampulla of Vater cancer, biliary tract cancer, or bile duct cancer. However, studies of pancreatic cancer, gallbladder cancer, or intrahepatic cholangiocarcinoma are excluded. 

In the Excel file, the titles required for analysis are in the "title" column, and the abstracts are in the "abstract" column. The following method will be applied, and 
the results will be compared with the answers provided by human experts. 

Criteria Approach:
Based on the following criteria, assign a classification of yes or no. An abstract should be classified as “yes” if it potentially meets all inclusion criteria, even if 
full confirmation requires the full text. However, if the abstract clearly fails to meet at least one inclusion criterion, it should be classified as “no.”Additionally, 
If the abstract is classified as ‘no’, document the reason for each classification for each query. 

Inclusion Criteria
1) clinical studies on extrahepatic cholangiocarcinoma, having comparative survival data according to use of adjuvant radiotherapy. 
2) inclusion of 10 or more patients who underwent radiotherapy; 
3) Each comparative arm (with and without radiotherapy) include 5 or more patients.

"""

# build_prompt: 한 편의 타이틀/초록을 붙여 JSON만 출력하게 지시
def build_prompt(title: str, abstract: str) -> str:
    return f"""{SCREENING_PROMPT}

Now, screen the following study and return ONLY a compact JSON object with keys "answer" and "reason".
- "answer" must be either "yes" or "no".
- If "answer" is "no", the "reason" should briefly state which inclusion criterion fails.
- If "answer" is "yes", the "reason" should briefly state which criteria appear potentially satisfied (even if full text is needed).

Title: {title}
Abstract: {abstract}

Return JSON only, no extra text.
"""

# JSON 파서(실수로 텍스트가 섞여도 JSON 추출 시도)
def _safe_json_parse(text: str):
    # 먼저 전체를 시도
    try:
        return json.loads(text)
    except Exception:
        pass
    # 중괄호 블록만 추출하여 재시도
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    # 실패 시 기본값
    return {"answer": "no", "reason": "Failed to parse JSON from model output."}

# GPT 평가 함수 (재현성/로그 강화)
def evaluate_with_chatgpt(title, abstract):
    prompt = build_prompt(title, abstract)

    def _call_chat(use_json=True):
        kwargs = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt},
            ],
            "temperature": TEMP,
            "top_p": TOP_P,
            "seed": SEED,
        }
        if use_json and JSON_MODE:
            kwargs["response_format"] = JSON_MODE
        return client.chat.completions.create(**kwargs)

    try:
        try:
            resp = _call_chat(use_json=True)   # 1차: JSON 모드 사용
        except Exception:
            resp = _call_chat(use_json=False)  # 2차: JSON 모드 없이 재시도

        content = resp.choices[0].message.content.strip()
        j = _safe_json_parse(content)

        ans = str(j.get("answer", "")).lower().strip()
        if ans not in {"yes", "no"}:
            ans = "no"
            j["reason"] = f'잘못된 answer 값. Raw: {content[:120]}...'

        meta = {
            "model_returned": getattr(resp, "model", MODEL_NAME),
            "created": getattr(resp, "created", None),
            "used_temperature": TEMP,
            "used_top_p": TOP_P,
        }
        if hasattr(resp, "usage") and resp.usage:
            meta.update({
                "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                "total_tokens": getattr(resp.usage, "total_tokens", None),
            })

        return {
            "answer": ans,
            "reason": j.get("reason", ""),
            "raw_content": content,
            **meta
        }

    except Exception as e:
        return {
            "answer": "no",
            "reason": f"Exception: {e}",
            "raw_content": None,
            "model_returned": MODEL_NAME,
            "created": None,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
        }

# NaN/비문자 안전 캐노니컬라이저
def _canonicalize(text) -> str:
    if isinstance(text, str):
        s = text
    else:
        s = "" if pd.isna(text) else str(text)
    s = s.replace("\r\n", "\n").replace("\r", "\n").strip()
    return "\n".join(ln.rstrip() for ln in s.split("\n"))

def _uid(title: str, abstract: str) -> str:
    import hashlib
    s = _canonicalize(title) + "\n" + _canonicalize(abstract)
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

# (재개) 이미 처리한 UID 로드
done_uids = set()
if OUT_JSONL.exists():
    with OUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                uid = rec.get("uid")
                if uid: done_uids.add(uid)
            except json.JSONDecodeError:
                pass

# 입력 로드 + 결측 정리
df = pd.read_excel(INPUT_XLSX)
assert "title" in df.columns and "abstract" in df.columns, "Excel에 'title'과 'abstract' 열이 필요합니다."
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")

# 행별 즉시 저장 실행
out_fh = OUT_JSONL.open("a", encoding="utf-8")
try:
    for row_idx, row in enumerate(tqdm(df.itertuples(index=False), total=len(df))):
        title = _canonicalize(getattr(row, "title", ""))
        abstract = _canonicalize(getattr(row, "abstract", ""))
        uid = _uid(title, abstract)

        # 이미 처리한 항목은 건너뜀(재개)
        if uid in done_uids:
            continue

        # 빈 초록은 스킵(원하면 호출하고 싶으면 이 if 블록을 제거)
        if abstract == "":
            rec = {
                "row_index": row_idx, "uid": uid,
                "title": title, "abstract": abstract,
                "ai_answer": "no",
                "ai_reason": "Empty abstract.",
                "response_time_sec": 0.0,
                "ai_prompt_tokens": 0,
                "ai_completion_tokens": 0,
            }
        else:
            # 호출
            start = time.time()
            try:
                result = evaluate_with_chatgpt(title, abstract)
                elapsed = round(time.time() - start, 3)
                rec = {
                    "row_index": row_idx, "uid": uid,
                    "title": title, "abstract": abstract,
                    "ai_answer": result.get("answer", "no"),
                    "ai_reason": result.get("reason", ""),
                    "response_time_sec": elapsed,
                    "ai_prompt_tokens": result.get("prompt_tokens"),
                    "ai_completion_tokens": result.get("completion_tokens"),
                    # 필요 시 원시 출력/메타 추가 가능
                    "raw_output": result.get("raw_content"),
                    "model_returned": result.get("model_returned"),
                    "created": result.get("created"),
                }
            except Exception as e:
                elapsed = round(time.time() - start, 3)
                rec = {
                    "row_index": row_idx, "uid": uid,
                    "title": title, "abstract": abstract,
                    "ai_answer": "no",
                    "ai_reason": f"Exception: {e}",
                    "response_time_sec": elapsed,
                    "ai_prompt_tokens": None,
                    "ai_completion_tokens": None,
                    "raw_output": None,
                    "model_returned": MODEL_NAME,
                    "created": None,
                }

        # 행 단위 즉시 저장 + 강제 flush
        out_fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out_fh.flush(); os.fsync(out_fh.fileno())
        done_uids.add(uid)
finally:
    out_fh.close()

# JSONL → 요약 테이블 → 엑셀 저장
records = []
with OUT_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass

# 현재 입력파일에 해당하는 기록만, 입력 순서대로
valid_uids = {
    _uid(_canonicalize(getattr(r, "title", "")),
         _canonicalize(getattr(r, "abstract", "")))
    for r in df.itertuples(index=False)
}
records = [r for r in records if r.get("uid") in valid_uids]
records.sort(key=lambda r: r.get("row_index", 10**12))

res_df = pd.DataFrame.from_records(records, columns=[
    "ai_answer","ai_reason","response_time_sec",
    "ai_prompt_tokens","ai_completion_tokens",
    "model_returned","created"
])
res_df.to_excel(OUT_XLSX, index=False)
print(f"처리 완료: {OUT_XLSX.name} 저장됨 (중간 체크포인트: {OUT_JSONL.name})")

# ---- run_meta.json 요약/검증 저장 ----
import json, time

# 1) 스냅샷 혼용 여부 확인
unique_models = sorted({ r.get("model_returned")
                         for r in records if r.get("model_returned") })
model_snapshot = unique_models[0] if unique_models else MODEL_NAME
mixed_models = (len(unique_models) > 1)

# 2) 응답 생성 시각 범위
created_vals = [r.get("created") for r in records if r.get("created") is not None]
time_window = {
    "first_created": min(created_vals) if created_vals else None,
    "last_created":  max(created_vals) if created_vals else None,
}

# 3) 토큰 합계(비용 계산도 원하면 포함)
total_prompt = int(sum((r.get("ai_prompt_tokens") or 0) for r in records))
total_completion = int(sum((r.get("ai_completion_tokens") or 0) for r in records))
# (선택) 비용: PRICING 딕트가 있으면 사용
PRICING = {
    "input_per_1M": INPUT_1M,    
    "output_per_1M": OUTPUT_1M,  
}
est_cost = (total_prompt/1_000_000)*PRICING["input_per_1M"] \
         + (total_completion/1_000_000)*PRICING["output_per_1M"]

# 4) 프롬프트 해시(이미 _sha256 함수가 있으면 재사용)
def _sha256(s): 
    import hashlib
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

run_meta = {
    "model_snapshot": model_snapshot,
    "mixed_models_detected": mixed_models,
    "all_models_seen": unique_models,           # 혼용 시 추적용
    "seed": SEED,
    "sampling_locked": True,                    # 이 스냅샷은 temperature/top_p 고정
    "effective_temperature": TEMP,
    "effective_top_p": TOP_P,
    "time_window": time_window,                 # epoch sec
    "tokens": {
        "prompt": total_prompt,
        "completion": total_completion,
        "total": total_prompt + total_completion
    },
    "estimated_total_cost_usd": round(est_cost, 4),
    "prompt_hashes": {
        "system_message_sha256": _sha256(system_message),
        "screening_prompt_sha256": _sha256(SCREENING_PROMPT),
        # "format_instructions_sha256": _sha256(FORMAT_INSTRUCTIONS),  # 쓰면 추가
    },
    "input_file": INPUT_XLSX,
    "generated_at_utc_epoch": int(time.time())
}

with open("./run_meta.json", "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print("run_meta.json 저장됨")

# records → 결과 프레임
res_df_full = pd.DataFrame.from_records(records)

# 붙일 컬럼만 추려서(반드시 row_index 포함)
res_keep = res_df_full[[
    "row_index","ai_answer","ai_reason","response_time_sec",
    "ai_prompt_tokens","ai_completion_tokens","model_returned","created"
]]

# 원본 df 옆에 병합 → 새 파일로 저장
df_with_results = (
    df.reset_index(drop=True)
      .reset_index(names="row_index")
      .merge(res_keep, on="row_index", how="left")
      .drop(columns=["row_index"])
)
df_with_results.to_excel("./add_with_results.xlsx", index=False)

100%|██████████| 696/696 [1:22:20<00:00,  7.10s/it]


처리 완료: results.xlsx 저장됨 (중간 체크포인트: results.jsonl)
run_meta.json 저장됨


### 1.2. NSCLC

In [ ]:
# NSCLC_Proba

import json, re, time, hashlib
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import os
from pathlib import Path

# ================== 설정 ==================
MODEL_NAME = "gpt-5-mini-2025-08-07"   
SEED = 12345
TEMP = None # 0 (gpt-4o)
TOP_P = None # 1 (gpt-4o)
JSON_MODE = {"type": "json_object"}  # JSON 강제
INPUT_1M = 0.25 # 2.5, 1.25, 0.25
OUTPUT_1M = 2.00 # 10.00, 10.00, 2.00
INPUT_XLSX = "./ELLMA1_NSCLC_250910_abstracts.xlsx"
OUT_JSONL  = Path("./results.jsonl")
OUT_XLSX   = Path("./results.xlsx")

client = OpenAI(api_key=API_KEY, timeout=60)
# =========================================
# model = 'gpt-4o-2024-11-20', 'gpt-5-2025-08-07', 'gpt-5-mini-2025-08-07'

# System message 변수 정의
system_message = """
You are a specialized assistant for experts conducting meta-analyses and developing clinical practice guidelines.
You support users in formulating key questions, conducting abstract screening, and performing data extraction during systematic reviews.
You emphasize evidence-based decision-making and strictly follow up-to-date research methodologies and guideline development processes.

You utilize frameworks such as PICO(TS) to structure research questions and apply methodologies like PRISMA and GRADE to assess the quality of evidence.
You also help optimize literature search strategies and refine inclusion and exclusion criteria based on the user's objectives.

You respond with precise terminology and accurate information. When needed, you can search for the latest research trends and suggest the most appropriate methodological approach according to the user's context and goals.
"""

# ======= 선별 규칙 설명 프롬프트 (확률/플래그/근거 포함) =======
SCREENING_PROMPT = """
Research Hypothesis and Objectives; This file is a list of abstracts for meta-analysis, aiming to identify studies that evaluate the oncologic benefit of local ablative treatment (LAT, e.g., surgery, radiotherapy, RFA), versus standard of care (e.g., systemic treatment or supportive care), in oligometastatic NSCLC patients.

Ontology = Definition of Key Terms and Links to Existing Terminology; LAT refers to local treatment including metastatic and/or primary sites (radiotherapy, surgery, radiofrequency ablation). LAT is closely aligned with local consolidative therapy (LCT). The control arm is a group without LAT (often systemic therapy or best supportive care). Oligometastases includes oligorecurrence, oligopersistence, and oligoresidual disease; some papers say “limited metastases.”

Data columns: "title" and "abstract". Your task is abstract-level screening.

Criteria:
Inclusion (all must be potentially met):
  I1) A comparative study with a LAT arm vs a standard-of-care (non-LAT) arm.
  I2) ≥10 oligometastatic NSCLC patients per arm.
  I3) OS or PFS reported as a primary endpoint (or clearly analyzable).

Exclusion:
  E1) Reviews, letters, errata, systematic reviews, preclinical.
  E2) Clinical studies that clearly fail any inclusion criterion.

Output policy:
- Return ONLY a compact JSON.
- Include a probabilistic "score" in [0.00, 1.00] indicating the likelihood the study should be INCLUDED for full-text screening under the above criteria.
- Calibrate using the rubric below. If evidence is uncertain/implicit, lower the score accordingly. Do not output >0.95 unless all 3 inclusion criteria are explicit.

Scoring rubric (anchor points):
  0.90–0.95: Explicit comparative LAT vs non-LAT; oligometastatic NSCLC; ≥10/arm stated or clearly implied; OS/PFS reported.
  0.70–0.85: Comparative LAT likely; oligometastatic NSCLC likely; sample size or endpoint partially unclear.
  0.50–0.65: Oligometastatic NSCLC and LAT mentioned but comparators/endpoints/sizes unclear.
  0.20–0.45: Likely fails ≥1 inclusion criterion, but some LAT/oligo signals present.
  0.05–0.15: Clearly a review/non-comparative/wrong population or endpoint.

JSON schema:
{
  "answer": "yes" | "no",
  "score": 0.00-1.00 (number with two decimals),
  "reason": "1-2 sentences justifying the score",
  "criteria_flags": {
    "has_comparator_lat_vs_control": true|false|"unclear",
    "n_per_arm_ge10": true|false|"unclear",
    "endpoint_os_or_pfs": true|false|"unclear",
    "population_oligomet_nsclc": true|false|"unclear",
    "study_type_clinical": true|false|"unclear"
  },
  "evidence_phrases": ["short supporting phrase 1", "phrase 2"],
  "threshold_hint": 0.70
}

Derive "answer" from the score using threshold_hint (score ≥ threshold_hint → "yes", else "no"). Keep "reason" concise. Prefer "unclear" over guessing when the abstract is ambiguous.
"""

# build_prompt: 확률/플래그 스키마 사용
def build_prompt(title: str, abstract: str) -> str:
    return f"""{SCREENING_PROMPT}

Now, screen the following study and return ONLY a compact JSON object following the schema above.

Title: {title}
Abstract: {abstract}

Return JSON only, no extra text.
"""

# JSON 파서(실수로 텍스트가 섞여도 JSON 추출 시도)
def _safe_json_parse(text: str):
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    # 실패 시 기본값
    return {
        "answer": "no",
        "reason": "Failed to parse JSON from model output.",
        "score": 0.0,
        "criteria_flags": {}
    }

def _to_bool_or_str(v):
    if isinstance(v, bool): return v
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"true","yes"}: return True
        if s in {"false","no"}: return False
        return "unclear"
    return "unclear"

def _to_float01(x, default=0.0):
    try:
        fx = float(x)
        if fx < 0: return 0.0
        if fx > 1: return 1.0
        return fx
    except Exception:
        return default

# GPT 평가 함수 (재현성/로그 강화)
def evaluate_with_chatgpt(title, abstract):
    prompt = build_prompt(title, abstract)

    def _call_chat(use_json=True):
        kwargs = {
            "model": MODEL_NAME,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt},
            ],
            # "temperature": TEMP,
            # "top_p": TOP_P,
            "seed": SEED,
        }
        if use_json and JSON_MODE:
            kwargs["response_format"] = JSON_MODE
        return client.chat.completions.create(**kwargs)

    try:
        try:
            resp = _call_chat(use_json=True)   # 1차: JSON 모드
        except Exception:
            resp = _call_chat(use_json=False)  # 2차: 일반 모드

        content = resp.choices[0].message.content.strip()
        j = _safe_json_parse(content)

        # 원본 answer
        answer_raw = str(j.get("answer", "")).lower().strip()
        if answer_raw not in {"yes","no"}:
            answer_raw = "no"

        # 점수/임계값 → 점수기반 판정
        score = _to_float01(j.get("score", 0.0))
        th = _to_float01(j.get("threshold_hint", 0.70), default=0.70)
        answer_from_score = "yes" if score >= th else "no"

        # 플래그 안전 파싱
        flags = j.get("criteria_flags", {}) or {}
        f_comp = _to_bool_or_str(flags.get("has_comparator_lat_vs_control"))
        f_narm = _to_bool_or_str(flags.get("n_per_arm_ge10"))
        f_ep   = _to_bool_or_str(flags.get("endpoint_os_or_pfs"))
        f_pop  = _to_bool_or_str(flags.get("population_oligomet_nsclc"))
        f_type = _to_bool_or_str(flags.get("study_type_clinical"))

        # 근거 구절
        evidence = j.get("evidence_phrases") or []
        if not isinstance(evidence, list): evidence = [str(evidence)]

        meta = {
            "model_returned": getattr(resp, "model", MODEL_NAME),
            "created": getattr(resp, "created", None),
            "used_temperature": TEMP,
            "used_top_p": TOP_P,
        }
        if hasattr(resp, "usage") and resp.usage:
            meta.update({
                "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                "total_tokens": getattr(resp.usage, "total_tokens", None),
            })

        return {
            "answer_raw": answer_raw,                 # 모델이 낸 원본 yes/no
            "answer_from_score": answer_from_score,   # 점수 기반 임계값 판정
            "threshold_hint": th,
            "score": round(score, 4),
            "reason": j.get("reason", ""),
            "criteria_flags": {
                "has_comparator_lat_vs_control": f_comp,
                "n_per_arm_ge10": f_narm,
                "endpoint_os_or_pfs": f_ep,
                "population_oligomet_nsclc": f_pop,
                "study_type_clinical": f_type,
            },
            "evidence_phrases": evidence,
            "raw_content": content,
            **meta
        }

    except Exception as e:
        return {
            "answer_raw": "no",
            "answer_from_score": "no",
            "threshold_hint": 0.70,
            "score": 0.0,
            "reason": f"Exception: {e}",
            "criteria_flags": {},
            "evidence_phrases": [],
            "raw_content": None,
            "model_returned": MODEL_NAME,
            "created": None,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
        }

# NaN/비문자 안전 캐노니컬라이저
def _canonicalize(text) -> str:
    if isinstance(text, str):
        s = text
    else:
        s = "" if pd.isna(text) else str(text)
    s = s.replace("\r\n", "\n").replace("\r", "\n").strip()
    return "\n".join(ln.rstrip() for ln in s.split("\n"))

def _uid(title: str, abstract: str) -> str:
    s = _canonicalize(title) + "\n" + _canonicalize(abstract)
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

# (재개) 이미 처리한 UID 로드
done_uids = set()
if OUT_JSONL.exists():
    with OUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                uid = rec.get("uid")
                if uid: done_uids.add(uid)
            except json.JSONDecodeError:
                pass

# 입력 로드 + 결측 정리
df = pd.read_excel(INPUT_XLSX)
assert "title" in df.columns and "abstract" in df.columns, "Excel에 'title'과 'abstract' 열이 필요합니다."
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")

# 이번 런에서 새로 처리한 결과를 담을 컨테이너
records = []

# 행별 즉시 저장 실행
out_fh = OUT_JSONL.open("a", encoding="utf-8")
try:
    for row_idx, row in enumerate(tqdm(df.itertuples(index=False), total=len(df))):
        title = _canonicalize(getattr(row, "title", ""))
        abstract = _canonicalize(getattr(row, "abstract", ""))
        uid = _uid(title, abstract)

        # 이미 처리한 항목은 건너뜀(재개)
        if uid in done_uids:
            continue

        # (빈 초록)
        if abstract == "":
            rec = {
                "row_index": row_idx, "uid": uid,
                "title": title, "abstract": abstract,
                "ai_answer_raw": "no",
                "ai_answer_from_score": "no",
                "ai_threshold_hint": 0.70,
                "ai_score": 0.0,
                "ai_reason": "Empty abstract.",
                "criteria_flags": {},
                "evidence_phrases": [],
                "evidence_joined": "",
                "response_time_sec": 0.0,
                "ai_prompt_tokens": 0,
                "ai_completion_tokens": 0,
                "model_returned": MODEL_NAME,
                "created": None,
                "raw_output": None,
            }
        else:
            start = time.time()
            try:
                result = evaluate_with_chatgpt(title, abstract)
                elapsed = round(time.time() - start, 3)
                flags = result.get("criteria_flags", {}) or {}
                ev = result.get("evidence_phrases", []) or []
                rec = {
                    "row_index": row_idx, "uid": uid,
                    "title": title, "abstract": abstract,
                    "ai_answer_raw": result.get("answer_raw", "no"),
                    "ai_answer_from_score": result.get("answer_from_score", "no"),
                    "ai_threshold_hint": result.get("threshold_hint", 0.70),
                    "ai_score": result.get("score", 0.0),
                    "ai_reason": result.get("reason", ""),
                    "criteria_flags": flags,
                    "flag_has_comparator": flags.get("has_comparator_lat_vs_control"),
                    "flag_n_per_arm_ge10": flags.get("n_per_arm_ge10"),
                    "flag_endpoint_os_or_pfs": flags.get("endpoint_os_or_pfs"),
                    "flag_population_oligo_nsclc": flags.get("population_oligomet_nsclc"),
                    "flag_study_type_clinical": flags.get("study_type_clinical"),
                    "evidence_phrases": ev,
                    "evidence_joined": " | ".join(ev),
                    "response_time_sec": elapsed,
                    "ai_prompt_tokens": result.get("prompt_tokens"),
                    "ai_completion_tokens": result.get("completion_tokens"),
                    "raw_output": result.get("raw_content"),
                    "model_returned": result.get("model_returned"),
                    "created": result.get("created"),
                }
            except Exception as e:
                elapsed = round(time.time() - start, 3)
                rec = {
                    "row_index": row_idx, "uid": uid,
                    "title": title, "abstract": abstract,
                    "ai_answer_raw": "no",
                    "ai_answer_from_score": "no",
                    "ai_threshold_hint": 0.70,
                    "ai_score": 0.0,
                    "ai_reason": f"Exception: {e}",
                    "criteria_flags": {},
                    "evidence_phrases": [],
                    "evidence_joined": "",
                    "response_time_sec": elapsed,
                    "ai_prompt_tokens": None,
                    "ai_completion_tokens": None,
                    "raw_output": None,
                    "model_returned": MODEL_NAME,
                    "created": None,
                }

        # 행 단위 즉시 저장 + 강제 flush
        out_fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out_fh.flush(); os.fsync(out_fh.fileno())
        done_uids.add(uid)

        # ✅ 이번 런 결과를 메모리에도 담기
        records.append(rec)
finally:
    out_fh.close()

# ✅ 만약 이번 런에서 새로 처리한 레코드가 하나도 없으면,
#    results.jsonl 전체를 읽어 현재 입력(df)에 해당하는 기록으로 records 재구성
if len(records) == 0 and OUT_JSONL.exists():
    valid_uids = {
        _uid(_canonicalize(getattr(r, "title", "")),
             _canonicalize(getattr(r, "abstract", "")))
        for r in df.itertuples(index=False)
    }
    latest_by_uid = {}
    with OUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            u = r.get("uid")
            if u in valid_uids:
                # 동일 uid가 여러 번 있으면 마지막 기록으로 갱신
                latest_by_uid[u] = r
    records = list(latest_by_uid.values())

# ---- run_meta.json 요약/검증 저장 ----
unique_models = sorted({ r.get("model_returned")
                         for r in records if r.get("model_returned") })
model_snapshot = unique_models[0] if unique_models else MODEL_NAME
mixed_models = (len(unique_models) > 1)

created_vals = [r.get("created") for r in records if r.get("created") is not None]
time_window = {
    "first_created": min(created_vals) if created_vals else None,
    "last_created":  max(created_vals) if created_vals else None,
}

total_prompt = int(sum((r.get("ai_prompt_tokens") or 0) for r in records))
total_completion = int(sum((r.get("ai_completion_tokens") or 0) for r in records))
PRICING = {"input_per_1M": INPUT_1M, "output_per_1M": OUTPUT_1M}
est_cost = (total_prompt/1_000_000)*PRICING["input_per_1M"] \
         + (total_completion/1_000_000)*PRICING["output_per_1M"]

def _sha256(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

run_meta = {
    "model_snapshot": model_snapshot,
    "mixed_models_detected": mixed_models,
    "all_models_seen": unique_models,
    "seed": SEED,
    "sampling_locked": True,
    "time_window": time_window,
    "tokens": {
        "prompt": total_prompt,
        "completion": total_completion,
        "total": total_prompt + total_completion
    },
    "estimated_total_cost_usd": round(est_cost, 4),
    "prompt_hashes": {
        "system_message_sha256": _sha256(system_message),
        "screening_prompt_sha256": _sha256(SCREENING_PROMPT),
    },
    "input_file": INPUT_XLSX,
    "generated_at_utc_epoch": int(time.time())
}

with open("./run_meta.json", "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print("run_meta.json 저장됨")

# records → 결과 프레임(원본 + 결과 병합본)
res_df_full = pd.DataFrame.from_records(records)

# 붙일 컬럼만 추려서(반드시 row_index 포함)
res_keep = res_df_full[[
    "row_index",
    "ai_answer_raw","ai_answer_from_score","ai_threshold_hint","ai_score",
    "ai_reason","response_time_sec",
    "ai_prompt_tokens","ai_completion_tokens",
    "flag_has_comparator","flag_n_per_arm_ge10",
    "flag_endpoint_os_or_pfs","flag_population_oligo_nsclc","flag_study_type_clinical",
    "evidence_joined",
    "model_returned","created"
]]

# 원본 df 옆에 병합 → 새 파일로 저장
df_with_results = (
    df.reset_index(drop=True)
      .reset_index(names="row_index")
      .merge(res_keep, on="row_index", how="left")
      .drop(columns=["row_index"])
)
df_with_results.to_excel("./add_with_results.xlsx", index=False)
print("처리 완료: add_with_results.xlsx 저장됨 (체크포인트: results.jsonl)")


100%|██████████| 431/431 [1:11:06<00:00,  9.90s/it]


run_meta.json 저장됨


## 2. Evaluation

In [11]:
# 3개(이상) 모델 결과 비교: 성능(Agreement/Sens/Spec/NPV/PPV/AUC) + 자원(response_time, cost)
# 전반 비교: Cochran's Q / Friedman
# 쌍대 비교: McNemar(이항) / Wilcoxon (Holm 보정)
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.metrics import confusion_matrix, roc_auc_score
from scipy.stats import binomtest, friedmanchisquare, wilcoxon
from statsmodels.stats.contingency_tables import cochrans_q

# =========================
# 사용자 설정
# =========================
xlsx_path   = "./NSCLC_Results.xlsx"
sheet_names = ["4.0", "5.0", "5.0-mini"]  # ← 여기에 3개 이상 시트명을 넣으세요.
human_expert = "final" # human expert 

# (토큰 비용 단가: 백만 토큰당 USD)
PROMPT_RATE = {"5.0": 1.250, "5.0-mini": 0.250, "4.0": 2.500}
COMP_RATE   = {"5.0":10.000, "5.0-mini": 2.000, "4.0": 10.000}

# =========================
# 유틸 함수
# =========================
def mcnemar_p(b, c):
    """McNemar exact p-value (binomial) for discordant pairs b vs c."""
    n = b + c
    if n == 0:
        return 1.0
    return binomtest(k=min(b, c), n=n, p=0.5, alternative="two-sided").pvalue

def holm_correction(pvals_dict):
    """Holm-Bonferroni: {pair: p} -> {pair: p_adj}"""
    items = sorted(pvals_dict.items(), key=lambda x: x[1])  # by raw p ascending
    m = len(items)
    # 1) 기본 보정
    adj_list = [min((m - i) * p, 1.0) for i, (_, p) in enumerate(items)]  # i=0..m-1
    # 2) 단조 비감소 보장 (앞으로 진행하며 누적 max)
    for i in range(1, m):
        adj_list[i] = max(adj_list[i], adj_list[i-1])
    # 3) 원 키로 되돌리기
    adj = {}
    for (k, _), p_adj in zip(items, adj_list):
        adj[k] = p_adj
    return adj

def format_p(p):
    return "<0.001" if p < 1e-3 else f"{p:.3f}"

def compute_metrics(df):
    """단일 모델 df에서 성능 지표 계산"""
    df = df.copy()
    df["expert_bin"] = (df[human_expert].str.strip().str.lower() == "yes").astype(int)
    df["gpt_bin"]    = (df["ai_answer"].str.strip().str.lower() == "yes").astype(int)

    tn, fp, fn, tp = confusion_matrix(df["expert_bin"], df["gpt_bin"], labels=[0,1]).ravel()
    agreement   = (df["expert_bin"] == df["gpt_bin"]).mean()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    npv         = tn / (tn + fn) if (tn + fn) else 0.0
    ppv         = tp / (tp + fp) if (tp + fp) else 0.0
    auc         = roc_auc_score(df["expert_bin"], df["gpt_bin"])  # 이진 예측으로 계산(참고용)

    return {
        "Agreement": round(agreement, 3),
        "Sensitivity": round(sensitivity, 3),
        "Specificity": round(specificity, 3),
        "NPV": round(npv, 3),
        "PPV": round(ppv, 3),
        "AUC": round(auc, 3),
    }

# =========================
# 1) 데이터 로딩/전처리
# =========================
dfs = {}
for name in sheet_names:
    df = pd.read_excel(xlsx_path, sheet_name=name).copy()
    # 공통 바이너리 라벨/예측
    df["expert_bin"] = (df[human_expert].str.strip().str.lower() == "yes").astype(int)
    df["gpt_bin"]    = (df["ai_answer"].str.strip().str.lower() == "yes").astype(int)
    # 시간/비용
    df["response_time_min"] = df["response_time_sec"] / 60.0
    pr = PROMPT_RATE.get(name, list(PROMPT_RATE.values())[0])
    cr = COMP_RATE.get(name,   list(COMP_RATE.values())[0])
    df["cost"] = (df["ai_prompt_tokens"]/1e6)*pr + (df["ai_completion_tokens"]/1e6)*cr
    dfs[name] = df

# (중요) 응답시간/비용 비교는 **같은 케이스 순서**가 모델 간 일치한다고 가정합니다.
# 케이스 ID가 있으면 merge하여 정렬 일치시키는 것을 권장합니다.

# =========================
# 2) 모델별 요약 지표
# =========================
summary_rows = []
for name, df in dfs.items():
    m = compute_metrics(df)
    m.update({
        "Model": name,
        "response_time_min_sum": round(df["response_time_min"].sum(), 3),
        "cost_sum": round(df["cost"].sum(), 3),
        "N": len(df),
    })
    summary_rows.append(m)

results_df = pd.DataFrame(summary_rows)[
    ["Model","N","Agreement","Sensitivity","Specificity","NPV","PPV","AUC","response_time_min_sum","cost_sum"]
].sort_values("Model")

# =========================
# 3) 전반 비교: Cochran’s Q (정확도/민감도/특이도)
# =========================
# 정확도(정답=1, 오답=0) 매트릭스
acc_mat = []
for name in sheet_names:
    df = dfs[name]
    acc_mat.append((df["expert_bin"] == df["gpt_bin"]).astype(int).to_numpy())

# 민감도(양성에서 TP=1, FN=0): 첫 시트의 양성 케이스 마스크로 통일
pos_mask = dfs[sheet_names[0]]["expert_bin"].to_numpy() == 1
sens_mat = []
for name in sheet_names:
    df = dfs[name]
    sens_mat.append((df.loc[pos_mask, "gpt_bin"] == 1).astype(int).to_numpy())

# 특이도(음성에서 TN=1, FP=0): 첫 시트의 음성 케이스 마스크로 통일
neg_mask = dfs[sheet_names[0]]["expert_bin"].to_numpy() == 0
spec_mat = []
for name in sheet_names:
    df = dfs[name]
    spec_mat.append((df.loc[neg_mask, "gpt_bin"] == 0).astype(int).to_numpy())

# 리스트(모델별 1D) -> 2D 배열(행=케이스, 열=모델)
acc_array  = np.column_stack(acc_mat)   # shape: (N_cases, K_models)
sens_array = np.column_stack(sens_mat)  # shape: (N_pos,   K_models)
spec_array = np.column_stack(spec_mat)  # shape: (N_neg,   K_models)

# Cochran's Q : 단일 2D 배열을 인자로 전달
q_acc_p  = cochrans_q(acc_array).pvalue  if acc_array.shape[1]  > 2 else np.nan
q_sens_p = cochrans_q(sens_array).pvalue if sens_array.shape[1] > 2 else np.nan
q_spec_p = cochrans_q(spec_array).pvalue if spec_array.shape[1] > 2 else np.nan

# =========================
# 4) 쌍대 비교: McNemar (정확도/민감도/특이도) + Holm 보정
# =========================
pairs = list(combinations(sheet_names, 2))

def pairwise_mcnemar(metric_mat, label="acc"):
    """metric_mat: list of 0/1 배열(모델 수 만큼). pairwise McNemar p 반환"""
    pvals = {}
    for a, b in pairs:
        i, j = sheet_names.index(a), sheet_names.index(b)
        a_vec = metric_mat[i]
        b_vec = metric_mat[j]
        # discordant
        b01 = np.sum((a_vec == 1) & (b_vec == 0))
        c10 = np.sum((a_vec == 0) & (b_vec == 1))
        pvals[(a,b)] = mcnemar_p(b01, c10)
    return pvals, holm_correction(pvals)

p_acc_raw,  p_acc_holm  = pairwise_mcnemar(acc_mat,  "acc")
p_sens_raw, p_sens_holm = pairwise_mcnemar(sens_mat, "sens")
p_spec_raw, p_spec_holm = pairwise_mcnemar(spec_mat, "spec")

def ptable(pdict):
    idx = sorted(set([a for a,_ in pdict.keys()] + [b for _,b in pdict.keys()]))
    dfp = pd.DataFrame(index=idx, columns=idx, data="")
    for (a,b), p in pdict.items():
        dfp.loc[a,b] = format_p(p)
        dfp.loc[b,a] = format_p(p)
    return dfp

pairwise_acc_p    = ptable(p_acc_raw)
pairwise_acc_padj = ptable(p_acc_holm)
pairwise_sens_p    = ptable(p_sens_raw)
pairwise_sens_padj = ptable(p_sens_holm)
pairwise_spec_p    = ptable(p_spec_raw)
pairwise_spec_padj = ptable(p_spec_holm)

# =========================
# 5) 응답시간/비용: Friedman + Wilcoxon(쌍대, Holm 보정)
# =========================
# 케이스별로 동일 순서라고 가정
rt_series   = [dfs[name]["response_time_min"].to_numpy() for name in sheet_names]
cost_series = [dfs[name]["cost"].to_numpy() for name in sheet_names]

friedman_rt_p   = friedmanchisquare(*rt_series).pvalue
friedman_cost_p = friedmanchisquare(*cost_series).pvalue

def pairwise_wilcoxon(series_list):
    pvals = {}
    for a, b in combinations(range(len(series_list)), 2):
        name_a, name_b = sheet_names[a], sheet_names[b]
        _, p = wilcoxon(series_list[a], series_list[b])
        pvals[(name_a, name_b)] = p
    return pvals, holm_correction(pvals)

rt_raw,   rt_holm   = pairwise_wilcoxon(rt_series)
cost_raw, cost_holm = pairwise_wilcoxon(cost_series)

pairwise_rt_p    = ptable(rt_raw)
pairwise_rt_padj = ptable(rt_holm)
pairwise_cost_p    = ptable(cost_raw)
pairwise_cost_padj = ptable(cost_holm)

# =========================
# 6) 출력/저장 (모든 결과를 한 시트에)
# =========================
with pd.ExcelWriter("results_3models_one_sheet.xlsx", engine="xlsxwriter") as writer:
    sheet_name = "AllResults"
    # 시트 먼저 생성해두고 writer.sheets에 등록
    ws = writer.book.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = ws

    state = {"row": 0}   # ← mutable 상태로 현재 행 관리
    padding = 2
    bold = writer.book.add_format({"bold": True})

    def write_block(title: str, df: pd.DataFrame):
        # 제목(엑셀 셀 직접 쓰기)
        ws.write(state["row"], 0, title, bold)
        state["row"] += 1
        # 표 쓰기
        df.to_excel(writer, sheet_name=sheet_name, startrow=state["row"], index=True)
        # 다음 블록 시작 위치 계산 (헤더 포함 + 패딩)
        state["row"] += (len(df) + 1 + padding)

    # 표 준비
    global_tests_df = pd.DataFrame({
        "Metric": ["Accuracy", "Sensitivity", "Specificity"],
        "Cochran_Q_p": [q_acc_p, q_sens_p, q_spec_p]
    }).set_index("Metric")

    # 작성 순서
    write_block("Summary metrics (per model)", results_df.set_index("Model"))
    write_block("Cochran's Q p-values (k>2)", global_tests_df)

    write_block("Pairwise McNemar p-values (Accuracy, raw)",         pairwise_acc_p)
    write_block("Pairwise McNemar p-values (Accuracy, Holm-adj)",    pairwise_acc_padj)
    write_block("Pairwise McNemar p-values (Sensitivity, raw)",      pairwise_sens_p)
    write_block("Pairwise McNemar p-values (Sensitivity, Holm-adj)", pairwise_sens_padj)
    write_block("Pairwise McNemar p-values (Specificity, raw)",      pairwise_spec_p)
    write_block("Pairwise McNemar p-values (Specificity, Holm-adj)", pairwise_spec_padj)

    write_block("Friedman p-values (response_time_min)",
                pd.DataFrame({"Friedman_p": [friedman_rt_p]}, index=["response_time_min"]))
    write_block("Friedman p-values (cost)",
                pd.DataFrame({"Friedman_p": [friedman_cost_p]}, index=["cost"]))

    write_block("Pairwise Wilcoxon p-values (response_time_min, raw)",      pairwise_rt_p)
    write_block("Pairwise Wilcoxon p-values (response_time_min, Holm-adj)", pairwise_rt_padj)
    write_block("Pairwise Wilcoxon p-values (cost, raw)",                    pairwise_cost_p)
    write_block("Pairwise Wilcoxon p-values (cost, Holm-adj)",               pairwise_cost_padj)

In [ ]:
# 5.0 vs. 5.0-mini (2개 비교)
import pandas as pd 
from sklearn.metrics import confusion_matrix, roc_auc_score
from scipy.stats import binomtest, shapiro, ttest_rel, wilcoxon
import numpy as np

# ----- 1) 데이터 불러오기 -----
df1 = pd.read_excel("./NSCLC_Results.xlsx", sheet_name="5.0")
df2 = pd.read_excel("./NSCLC_Results.xlsx", sheet_name="5.0-mini")

file_list   = [df1, df2]
sheet_names = ["5.0", "5.0-mini"]

# ----- 2) 성능 지표 계산 -----
results = []
for file, name in zip(file_list, sheet_names):
    df = file.copy()
    df["expert_bin"] = (df["human expert"].str.strip().str.lower() == "yes").astype(int) # human expert vs. final
    df["gpt_bin"]    = (df["ai_answer"].str.strip().str.lower() == "yes").astype(int)

    tn, fp, fn, tp = confusion_matrix(df["expert_bin"], df["gpt_bin"], labels=[0,1]).ravel()

    agreement   = (df["expert_bin"] == df["gpt_bin"]).mean()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0
    npv         = tn / (tn + fn) if (tn + fn) else 0
    ppv         = tp / (tp + fp) if (tp + fp) else 0
    roc_auc     = roc_auc_score(df["expert_bin"], df["gpt_bin"])

    results.append({
        "Model": name,
        "Agreement": round(agreement, 3),
        "Sensitivity": round(sensitivity, 3),
        "Specificity": round(specificity, 3),
        "NPV": round(npv, 3),
        "PPV": round(ppv, 3),
        "AUC": round(roc_auc, 3),
    })

results_df = pd.DataFrame(results)

# ----- 3) 맥네마 p-value 계산 -----
def mcnemar_p(b, c):
    n = b + c
    if n == 0:
        return 1.0
    return binomtest(k=min(b, c), n=n, p=0.5, alternative="two-sided").pvalue

# 이진 벡터 준비
df1_bin, df2_bin = df1.copy(), df2.copy()
df1_bin["expert_bin"] = (df1_bin["human expert"].str.strip().str.lower() == "yes").astype(int)
df1_bin["gpt_bin"]    = (df1_bin["ai_answer"].str.strip().str.lower() == "yes").astype(int)
df2_bin["expert_bin"] = (df2_bin["human expert"].str.strip().str.lower() == "yes").astype(int)
df2_bin["gpt_bin"]    = (df2_bin["ai_answer"].str.strip().str.lower() == "yes").astype(int)

# Accuracy
corr1 = (df1_bin["expert_bin"] == df1_bin["gpt_bin"]).astype(int)
corr2 = (df2_bin["expert_bin"] == df2_bin["gpt_bin"]).astype(int)
b_acc = ((corr1 == 1) & (corr2 == 0)).sum()
c_acc = ((corr1 == 0) & (corr2 == 1)).sum()
p_acc = mcnemar_p(b_acc, c_acc)

# Sensitivity
pos_mask = (df1_bin["expert_bin"] == 1)
pred1_pos, pred2_pos = df1_bin.loc[pos_mask, "gpt_bin"], df2_bin.loc[pos_mask, "gpt_bin"]
b_sens = ((pred1_pos == 1) & (pred2_pos == 0)).sum()
c_sens = ((pred1_pos == 0) & (pred2_pos == 1)).sum()
p_sens = mcnemar_p(b_sens, c_sens)

# Specificity
neg_mask = (df1_bin["expert_bin"] == 0)
pred1_neg, pred2_neg = df1_bin.loc[neg_mask, "gpt_bin"], df2_bin.loc[neg_mask, "gpt_bin"]
b_spec = ((pred1_neg == 0) & (pred2_neg == 1)).sum()
c_spec = ((pred1_neg == 1) & (pred2_neg == 0)).sum()
p_spec = mcnemar_p(b_spec, c_spec)

# ----- 4) 시간/비용 계산 및 통계 -----
PROMPT_RATE_5, COMPLETION_RATE_5 = 1.250, 10.000
PROMPT_RATE_MINI, COMPLETION_RATE_MINI = 0.250, 2.000

df1["response_time_min"] = df1["response_time_sec"] / 60.0
df2["response_time_min"] = df2["response_time_sec"] / 60.0

df1["cost"] = (df1["ai_prompt_tokens"]/1e6)*PROMPT_RATE_5 + (df1["ai_completion_tokens"]/1e6)*COMPLETION_RATE_5
df2["cost"] = (df2["ai_prompt_tokens"]/1e6)*PROMPT_RATE_MINI + (df2["ai_completion_tokens"]/1e6)*COMPLETION_RATE_MINI

def choose_test(x, y, alpha=0.05):
    diff = x - y
    shapiro_p = shapiro(diff)[1] if len(diff) <= 5000 else 1.0
    if shapiro_p > alpha:
        _, p = ttest_rel(x, y, nan_policy="omit")
        method = "Paired t-test"
    else:
        _, p = wilcoxon(x, y)
        method = "Wilcoxon signed-rank"
    return method, p

# response_time
m_time, p_time = choose_test(df1["response_time_min"].to_numpy(), df2["response_time_min"].to_numpy())
# cost
m_cost, p_cost = choose_test(df1["cost"].to_numpy(), df2["cost"].to_numpy())

def format_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

# ----- 5) p-value 행 추가 -----
p_row = {
    "Model": "p-value",
    "Agreement": format_p(p_acc),
    "Sensitivity": format_p(p_sens),
    "Specificity": format_p(p_spec),
    "NPV": "",
    "PPV": "",
    "AUC": "",
    "response_time_min": format_p(p_time),
    "cost": format_p(p_cost),
}

# 성능 + 리소스 합계 붙이기
results_df["response_time_min"] = [round(df1["response_time_min"].sum(), 3), round(df2["response_time_min"].sum(), 3)]
results_df["cost"] = [round(df1["cost"].sum(), 3), round(df2["cost"].sum(), 3)]
results_df = pd.concat([results_df, pd.DataFrame([p_row])], ignore_index=True)

# ----- 6) 저장 및 출력 -----
print(results_df)
print("\n[통계 방법]")
print(f"response_time_min → {m_time}")
print(f"cost → {m_cost}")

results_df.to_excel("results.xlsx", index=False)


      Model Agreement Sensitivity Specificity    NPV    PPV    AUC  \
0       5.0     0.977       0.588       0.993  0.983  0.769   0.79   
1  5.0-mini     0.981       0.706       0.993  0.988    0.8  0.849   
2   p-value     0.625       0.500       1.000                        

  response_time_min    cost  
0             65.88   2.567  
1            47.547   0.385  
2            <0.001  <0.001  

[통계 방법]
response_time_min → Wilcoxon signed-rank
cost → Wilcoxon signed-rank
